# SLIMA — Histologic Lung Patterns (6×H100, Accelerate v5)
**Notebook version: 15 Nov 2025**

- OPTIMIZE FOR F1. Best trial: {'S_SCALE': 15.791641099884055, 'M_MARGIN': 0.34510989013863635, 'TAU': 0.38164560360000555}
- Fix: Excel uses `tile_id` for IDs and `pattern` for labels.
- Fix: Avoids triggering single‑GPU run by guarding the CLI entry inside notebooks.
- Use the last cell to launch **6 GPUs** via `accelerate.notebook_launcher`.
- Robust pairing: recursively indexes *.mat in MASK_DIR by stem and normalized key.
- Excel labels: map by basename or tile_id; ignores absolute paths.
- ROI-guided options:
    * USE_MASK_AS_CHANNEL=True -> 4-channel input (RGB+mask); adjusts first conv.
    * USE_ROI_CROP=True        -> crops to bbox of mask with padding.
- FuzzyArcLoss (tau, margin, scale) + class-weighted CE.
- Jupyter: use launch_from_notebook(num_processes=6).
- CLI:    `accelerate launch <this_script>.py`  (guarded to not trigger from notebook)
Requirements:
    pip install accelerate torch torchvision scikit-learn pillow scipy h5py

In [1]:

import os, re, json, time, random, sys
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [2]:

try:
    from scipy.io import loadmat
except Exception:
    loadmat = None
try:
    import h5py
except Exception:
    h5py = None

from accelerate import Accelerator, notebook_launcher


In [3]:
import os, torch
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
torch.set_default_dtype(torch.float32)   # solo por claridad; FP32 ya es el default


In [4]:

# ---------------------
# Default CONFIG
# ---------------------
ROOT_DIR  = "/home/rapids/notebooks/slima/Zenodo_Anorak_original"
IMAGE_DIR = f"{ROOT_DIR}/image"
MASK_DIR  = f"{ROOT_DIR}/mask"
# UPDATED to your new file (9 Nov)
XLS_PATH  = "/home/rapids/notebooks/slima/overlay_index ver 9 nov 2025.xlsx"

OUT_DIR   = "/home/rapids/notebooks/slima/outputs/anorak_roi_acc6_v5"

# Histologic classes to include (case-insensitive)
INCLUDE_PATTERNS = "lepidic,acinar,papillary,micropapillary,solid,mucinous"

# Model / train
ARCH = "resnet101"         # or "resnet101"
EMBED_DIM = 512
EPOCHS    = 120
BATCH_SIZE = 32           # per GPU
LR = 1e-3
WEIGHT_DECAY = 1e-4
IMG_SIZE = 384
SEED = 42
VAL_SIZE = 0.2
NUM_WORKERS = 6
GRAD_ACCUM_STEPS = 1

# FuzzyArc params
#Best trial: {'S_SCALE': 15.791641099884055, 'M_MARGIN': 0.34510989013863635, 'TAU': 0.38164560360000555}
S_SCALE = 15.791641099884055
M_MARGIN = 0.34510989013863635
TAU = 0.38164560360000555

# ROI-guided
USE_MASK_AS_CHANNEL = True   # 4-channel (RGB+mask)
USE_ROI_CROP = True         # crop to bbox of mask
ROI_PADDING = 24
MASK_THRESH = 0.5

# Accelerate / distributed
NUM_PROCESSES = 6            # H100 x 6
MIXED_PRECISION = "no"


In [5]:

torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# ---------------------
# Utils
# ---------------------
IMG_EXTS = {".png",".jpg",".jpeg",".tif",".tiff"}


In [6]:

def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def list_images(d: str) -> List[Path]:
    out = []
    base = Path(d)
    for p in base.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            out.append(p)
    return sorted(out)

def list_mats(d: str) -> List[Path]:
    return sorted([p for p in Path(d).rglob("*.mat") if p.is_file()])

def normalize_key(stem: str) -> str:
    # lower + strip common suffixes at end + collapse separators
    s = stem.lower()
    s = re.sub(r"([_-])(mask|seg|roi|label|overlay)([_-]?\d+)?$", "", s)
    s = re.sub(r"[ \t\-_]+", "_", s).strip("_")
    return s

def build_mat_index(mask_dir: str) -> Dict[str, str]:
    """Index .mat by multiple keys: exact stem, lower stem, normalized stem."""
    idx = {}
    for p in list_mats(mask_dir):
        stem = p.stem
        # choose the largest file per key (heuristic) if collision
        for key in (stem, stem.lower(), normalize_key(stem)):
            if key not in idx or p.stat().st_size > Path(idx[key]).stat().st_size:
                idx[key] = str(p)
    return idx

def pair_images_with_masks(image_dir: str, mask_dir: str) -> pd.DataFrame:
    imgs = list_images(image_dir)
    mat_index = build_mat_index(mask_dir)

    rows, unmatched = [], []
    for ip in imgs:
        stem = ip.stem
        key_norm = normalize_key(stem)
        mp = mat_index.get(stem) or mat_index.get(stem.lower()) or mat_index.get(key_norm)
        if mp:
            rows.append({"image_path": str(ip), "mask_path": mp, "base": stem, "base_norm": key_norm})
        else:
            unmatched.append(ip.name)

    print(f"[PAIRING] matched={len(rows)}  unmatched={len(unmatched)}")
    if unmatched[:10]:
        print("[UNMATCHED examples] first 10:", unmatched[:10])
    return pd.DataFrame(rows)


In [7]:

# ---------------------
# Excel labels
# ---------------------
def read_labels_from_xls(xls_path: str) -> Tuple[Dict[str, str], str]:
    """
    Returns (map, label_col). Supports the xls format shown in your screenshot:
    columns: tile_id, pattern, color_name, color_rgb, color_hex, color_swatch, pixels, area_pct
    """
    df = pd.read_excel(xls_path)
    # label column (which contains classes)
    label_candidates = ["pattern","label","class","type","luad_pattern","histologic_pattern","pattern_type"]
    file_candidates  = ["tile_id","overlay_file","image_file","file","path","filepath","filename","name","base","stem","image_id","tile"]
    label_col = next((c for c in label_candidates if c in df.columns), None)
    if label_col is None:
        raise ValueError(f"No label column among {label_candidates} in {xls_path}. Columns found: {df.columns.tolist()}")
    file_col = next((c for c in file_candidates if c in df.columns), None)
    if file_col is None:
        # If still nothing, try index as tile_id-like
        raise ValueError(f"No file-id column among {file_candidates} in {xls_path}. Columns found: {df.columns.tolist()}")
    m = {}
    for _, r in df.iterrows():
        f = str(r[file_col])
        stem = Path(f).stem  # if tile_id already a bare stem, this is the same
        lbl = str(r[label_col]).strip()
        # Accept both raw tile_id and normalized keys
        m[stem] = lbl
        m[normalize_key(stem)] = lbl
        m[stem.lower()] = lbl
    return m, label_col



In [8]:
# ---------------------
# .mat loader
# ---------------------
MAT_EXCLUDE = {"__header__","__version__","__globals__"}
MAT_PRIOR   = ["mask","Mask","BW","bw","label","Label","roi","ROI","seg","Seg","segmentation"]

def _load_mat_any(path: str) -> np.ndarray:
    # Try scipy (v7)
    if loadmat is not None:
        try:
            d = loadmat(path)
            for k in MAT_PRIOR:
                if k in d and isinstance(d[k], np.ndarray) and d[k].ndim >= 2:
                    return d[k]
            # pick largest 2D/3D array
            best, sz = None, -1
            for k, v in d.items():
                if k in MAT_EXCLUDE: continue
                if isinstance(v, np.ndarray) and v.ndim >= 2:
                    s = np.prod(v.shape[:2])
                    if s > sz: best, sz = v, s
            if best is not None:
                return best
        except Exception:
            pass
    # Try h5py (v7.3)
    if h5py is not None:
        try:
            with h5py.File(path, "r") as f:
                best, sz = None, -1
                def visit(name, obj):
                    nonlocal best, sz
                    if isinstance(obj, h5py.Dataset) and obj.ndim >= 2:
                        s = np.prod(obj.shape[:2])
                        if s > sz: best, sz = obj, s
                f.visititems(visit)
                if best is not None:
                    return np.array(best[()])
        except Exception:
            pass
    raise RuntimeError(f"No readable 2D/3D dataset found in .mat: {path}")

def load_mask_from_mat(path: str, thresh: float = 0.5) -> Image.Image:
    arr = _load_mat_any(path)
    arr = np.array(arr)
    while arr.ndim > 2 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    if arr.ndim > 2:
        # choose first channel or argmax if many channels
        arr = arr[..., 0] if arr.shape[-1] <= 4 else np.argmax(arr, axis=-1)
    mask = (arr.astype(np.float32) > 0).astype(np.uint8) * 255
    return Image.fromarray(mask, mode="L")



In [9]:
# ---------------------
# Dataset
# ---------------------
class HistMaskDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, label_col: str, label2id: Dict[str, int],
                 img_size: int = 384, aug: bool = True,
                 use_mask_as_channel: bool = True,
                 use_roi_crop: bool = False,
                 roi_padding: int = 16,
                 mask_thresh: float = 0.5):
        self.rows = rows.reset_index(drop=True)
        self.label_col = label_col
        self.label2id = label2id
        self.img_size = img_size
        self.aug = aug
        self.use_mask_as_channel = use_mask_as_channel
        self.use_roi_crop = use_roi_crop
        self.roi_padding = roi_padding
        self.mask_thresh = mask_thresh
        self.cj = transforms.ColorJitter(0.2, 0.2, 0.2, 0.05)

    def __len__(self): return len(self.rows)

    def _roi_crop(self, img: Image.Image, msk: Image.Image):
        m = np.array(msk) > 0
        if not m.any(): return img, msk
        ys, xs = np.where(m)
        y0 = max(0, ys.min() - self.roi_padding)
        y1 = min(m.shape[0], ys.max()+1 + self.roi_padding)
        x0 = max(0, xs.min() - self.roi_padding)
        x1 = min(m.shape[1], xs.max()+1 + self.roi_padding)
        return img.crop((x0,y0,x1,y1)), msk.crop((x0,y0,x1,y1))

    def _joint(self, img: Image.Image, msk: Image.Image):
        if self.aug:
            i, j, h, w = transforms.RandomResizedCrop.get_params(img, scale=(0.6,1.0), ratio=(0.9,1.1))
            img = TF.resized_crop(img, i,j,h,w, size=[self.img_size,self.img_size], interpolation=InterpolationMode.BILINEAR)
            msk = TF.resized_crop(msk, i,j,h,w, size=[self.img_size,self.img_size], interpolation=InterpolationMode.NEAREST)
            if random.random()<0.5: img = TF.hflip(img); msk = TF.hflip(msk)
            if random.random()<0.5: img = TF.vflip(img); msk = TF.vflip(msk)
            angle = random.uniform(-15,15)
            img = TF.rotate(img, angle, interpolation=InterpolationMode.BILINEAR)
            msk = TF.rotate(msk, angle, interpolation=InterpolationMode.NEAREST)
            if random.random() < 0.3: img = TF.gaussian_blur(img, kernel_size=3)
            if random.random() < 0.2: img = TF.adjust_sharpness(img, sharpness_factor=2.0)
            if random.random() < 0.2: img = TF.posterize(img, bits=random.choice([4,5,6]))
        else:
            img = img.resize((self.img_size,self.img_size), resample=Image.BILINEAR)
            msk = msk.resize((self.img_size,self.img_size), resample=Image.NEAREST)
        return img, msk

    def __getitem__(self, idx: int):
        r = self.rows.iloc[idx]
        img = Image.open(r["image_path"]).convert("RGB")
        msk = load_mask_from_mat(r["mask_path"])

        if self.use_roi_crop:
            img, msk = self._roi_crop(img, msk)

        img, msk = self._joint(img, msk)
        if self.aug:
            img = self.cj(img)

        img_t = TF.to_tensor(img)
        img_t = TF.normalize(img_t, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        msk_t = (TF.to_tensor(msk) > self.mask_thresh).float()

        x = torch.cat([img_t, msk_t], dim=0) if self.use_mask_as_channel else img_t
        y = self.label2id[str(r[self.label_col])]
        return x, y, r["image_path"]


In [10]:

# ---------------------
# FuzzyArcLoss
# ---------------------
class FuzzyArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50, tau=0.10):
        super().__init__()
        self.s, self.m, self.tau = float(s), float(m), float(tau)
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, features, labels):
        x = F.normalize(features); W = F.normalize(self.weight).to(features.device)
        cos = (x @ W.t()).clamp(-1,1)
        idx = torch.arange(x.size(0), device=x.device)
        cos_y = cos[idx, labels]
        mu = torch.where(torch.abs(cos_y) >= self.tau, torch.abs(cos_y), torch.ones_like(cos_y))
        m_eff = self.m * mu
        cos_m, sin_m = torch.cos(m_eff), torch.sin(m_eff)
        sin_t = torch.sqrt((1 - cos_y**2).clamp(0,1))
        cos_theta_m = cos_y * cos_m - sin_t * sin_m
        logits = cos * self.s
        logits[idx, labels] = cos_theta_m * self.s
        return logits

class FuzzyArcLoss(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50, tau=0.10, ce_weight=None):
        super().__init__()
        self.head = FuzzyArcMarginProduct(in_features, out_features, s=s, m=m, tau=tau)
        self.ce = nn.CrossEntropyLoss(weight=ce_weight)
    def forward(self, feats, labels):
        logits = self.head(feats, labels)
        return self.ce(logits, labels), logits


In [11]:

# ---------------------
# Backbone
# ---------------------
def build_backbone(arch="resnet50", pretrained=True, embed_dim=512, in_channels=3):
    if arch=="resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    elif arch=="resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2 if pretrained else None)
    else:
        raise ValueError(arch)
    if in_channels != 3:
        old = m.conv1
        m.conv1 = nn.Conv2d(in_channels, old.out_channels, kernel_size=old.kernel_size,
                            stride=old.stride, padding=old.padding, bias=False)
        with torch.no_grad():
            m.conv1.weight[:, :3] = old.weight
            if in_channels > 3:
                mean_rgb = old.weight.mean(dim=1, keepdim=True)
                m.conv1.weight[:, 3:in_channels] = mean_rgb.repeat(1, in_channels-3, 1, 1)
    in_dim = m.fc.in_features
    m.fc = nn.Identity()
    head = nn.Linear(in_dim, embed_dim, bias=False)
    net = nn.Sequential(m, nn.BatchNorm1d(in_dim), head, nn.BatchNorm1d(embed_dim))
    return net, embed_dim

# ---------------------
# Scan & sanity (optional to run before training)
# ---------------------
def scan_and_sanity():
    imgs = list_images(IMAGE_DIR); mats = list_mats(MASK_DIR)
    print(f"[SCAN] images={len(imgs)} mats={len(mats)}")
    for ip in imgs[:5]:
        print(ip.name, "->", normalize_key(ip.stem))
    df = pair_images_with_masks(IMAGE_DIR, MASK_DIR)
    return df


In [12]:
class FocalCE(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("alpha", None if alpha is None else torch.tensor(alpha, dtype=torch.float32))
    def forward(self, logits, target):
        # logits: [B,C], target: [B]
        logp = F.log_softmax(logits, dim=1)
        p    = logp.exp()
        pt   = p[torch.arange(logits.size(0), device=logits.device), target]
        loss = -(1 - pt).pow(self.gamma) * logp[torch.arange(logits.size(0), device=logits.device), target]
        if self.alpha is not None:
            at = self.alpha[target]
            loss = loss * at
        return loss.mean()


In [13]:

# ---------------------
# Training function (Accelerate)
# ---------------------
def training_function():
    accelerator = Accelerator(mixed_precision=MIXED_PRECISION, gradient_accumulation_steps=GRAD_ACCUM_STEPS)
    if accelerator.is_main_process:
        ensure_dir(OUT_DIR)
    accelerator.print(f"Running on {accelerator.state.num_processes} GPUs; mp={accelerator.mixed_precision}")

    set_seed(SEED)

    # 1) Pair images and masks
    imgs_all = list_images(IMAGE_DIR)
    mats_all = list_mats(MASK_DIR)
    accelerator.print(f"[SCAN] images={len(imgs_all)} mats={len(mats_all)}")
    df = pair_images_with_masks(IMAGE_DIR, MASK_DIR)
    if df.empty:
        raise RuntimeError("No image↔mask pairs found. Check names/rutas or adjust normalize_key().")

    # 2) Map labels from Excel
    label_map, label_col = read_labels_from_xls(XLS_PATH)
    df[label_col] = df["base"].map(label_map)
    miss = df[label_col].isna()
    if miss.any():
        df.loc[miss, label_col] = df.loc[miss, "base_norm"].map(label_map)
    df = df[~df[label_col].isna()].copy()
    accelerator.print(f"[LABELS] after XLS join: {len(df)} rows")

    # 3) Filter classes
    inc = {p.strip().lower() for p in INCLUDE_PATTERNS.split(",") if p.strip()}
    df[label_col] = df[label_col].astype(str)
    df = df[df[label_col].str.lower().isin(inc)].copy()
    accelerator.print(f"[FILTER] kept={len(df)} over classes={sorted(list(inc))}")
    if df.empty:
        raise RuntimeError("After filtering by classes, dataset is empty. Check INCLUDE_PATTERNS and Excel labels.")

    # 4) Label maps
    classes = sorted(df[label_col].unique().tolist())
    label2id = {c:i for i,c in enumerate(classes)}
    id2label = {v:k for k,v in label2id.items()}
    if accelerator.is_main_process:
        print("[CLASSES]", label2id)

    # 5) Split & weights
    train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df[label_col])
    counts = Counter(train_df[label_col])
    # weights inversely proportional to class count
    ce_w = torch.tensor([1.0 / counts[id2label[i]] for i in range(len(id2label))], dtype=torch.float32, device=accelerator.device)

    # 6) Datasets & loaders
    in_ch = 4 if USE_MASK_AS_CHANNEL else 3
    train_ds = HistMaskDataset(train_df, label_col, label2id, IMG_SIZE, True,  USE_MASK_AS_CHANNEL, USE_ROI_CROP, ROI_PADDING, MASK_THRESH)
    val_ds   = HistMaskDataset(val_df,   label_col, label2id, IMG_SIZE, False, USE_MASK_AS_CHANNEL, USE_ROI_CROP, ROI_PADDING, MASK_THRESH)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # 7) Model / loss / optim
    model, _  = build_backbone(ARCH, pretrained=True, embed_dim=EMBED_DIM, in_channels=in_ch)
    #loss_head = FuzzyArcLoss(EMBED_DIM, len(label2id), s=S_SCALE, m=M_MARGIN, tau=TAU, ce_weight=ce_w)
    counts_tr = Counter(train_df[label_col])
    alpha = [1.0 / counts_tr[id2label[i]] for i in range(len(id2label))]
    alpha = (np.array(alpha) / np.sum(alpha)).tolist()

    loss_head = FuzzyArcLoss(EMBED_DIM, len(label2id), s=S_SCALE, m=M_MARGIN, tau=TAU, ce_weight=None)
    loss_head.ce = FocalCE(alpha=alpha, gamma=2.0)
    opt = torch.optim.AdamW(list(model.parameters())+list(loss_head.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5, min_lr=1e-6)

    model, loss_head, opt, train_ld, val_ld = accelerator.prepare(model, loss_head, opt, train_ld, val_ld)

    best_f1, best_path = -1.0, os.path.join(OUT_DIR, "best_model_roi_acc6_v3.pth")

    for epoch in range(1, EPOCHS+1):
        t0 = time.time()
        # ---- train
        model.train()
        y_t, y_p, tot = [], [], 0.0
        for x, y, _ in train_ld:
            feats = model(x)
            loss, logits = loss_head(feats, y)
            accelerator.backward(loss)
            opt.step(); opt.zero_grad(set_to_none=True)
            tot += loss.detach().float() * x.size(0)
            pred = torch.argmax(logits.detach(), 1)
            y_t.append(accelerator.gather(y).cpu()); y_p.append(accelerator.gather(pred).cpu())

        y_t = torch.cat(y_t).numpy(); y_p = torch.cat(y_p).numpy()
        tr_acc = accuracy_score(y_t, y_p); tr_f1 = f1_score(y_t, y_p, average="macro")
        tr_loss = (tot.item() / max(1, len(train_ld.dataset)))

        # ---- val
        model.eval()
        y_t, y_p, tot = [], [], 0.0
        with torch.no_grad():
            for x, y, _ in val_ld:
                feats = model(x)
                loss, logits = loss_head(feats, y)
                tot += loss.detach().float() * x.size(0)
                pred = torch.argmax(logits, 1)
                y_t.append(accelerator.gather(y).cpu()); y_p.append(accelerator.gather(pred).cpu())

        y_t = torch.cat(y_t).numpy(); y_p = torch.cat(y_p).numpy()
        va_acc = accuracy_score(y_t, y_p); va_f1 = f1_score(y_t, y_p, average="macro")
        va_loss = (tot.item() / max(1, len(val_ld.dataset)))
        sch.step(va_f1)

        accelerator.print(f"[{epoch:03d}] tr_loss={tr_loss:.4f} acc={tr_acc:.4f} f1={tr_f1:.4f} | "
                          f"val_loss={va_loss:.4f} acc={va_acc:.4f} f1={va_f1:.4f} | {time.time()-t0:.1f}s")

        if accelerator.is_main_process and va_f1 > best_f1:
            best_f1 = va_f1
            state = {
                "epoch": epoch,
                "model_state": accelerator.unwrap_model(model).state_dict(),
                "head_state": accelerator.unwrap_model(loss_head).state_dict(),
                "label2id": label2id, "id2label": {v:k for k,v in label2id.items()},
                "val_f1": va_f1,
                "config": {
                    "ROOT_DIR": ROOT_DIR, "IMAGE_DIR": IMAGE_DIR, "MASK_DIR": MASK_DIR,
                    "XLS_PATH": XLS_PATH, "OUT_DIR": OUT_DIR, "ARCH": ARCH, "EMBED_DIM": EMBED_DIM,
                    "EPOCHS": EPOCHS, "BATCH_SIZE": BATCH_SIZE, "LR": LR, "WEIGHT_DECAY": WEIGHT_DECAY,
                    "IMG_SIZE": IMG_SIZE, "SEED": SEED, "VAL_SIZE": VAL_SIZE, "NUM_WORKERS": NUM_WORKERS,
                    "S_SCALE": S_SCALE, "M_MARGIN": M_MARGIN, "TAU": TAU, "INCLUDE_PATTERNS": INCLUDE_PATTERNS,
                    "MIXED_PRECISION": accelerator.mixed_precision, "USE_MASK_AS_CHANNEL": USE_MASK_AS_CHANNEL,
                    "USE_ROI_CROP": USE_ROI_CROP, "ROI_PADDING": ROI_PADDING, "MASK_THRESH": MASK_THRESH
                }
            }
            torch.save(state, best_path)
            accelerator.print(f"[OK] saved best -> {best_path} (f1={va_f1:.4f})")

    if accelerator.is_main_process:
        names = [id for id in range(len(id2label))]
        rep = classification_report(y_t, y_p, target_names=[id2label[i] for i in names])
        with open(os.path.join(OUT_DIR, "val_classification_report.txt"), "w") as f:
            f.write(rep)
        print("\n=== Validation classification report ===\n", rep)



In [14]:
import optuna
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import DataLoader
from accelerate import Accelerator


In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score


In [16]:

# Helper function for computing logits without margin for TTA
def _logits_nomargin(feats, head):
    W = F.normalize(head.weight)
    feats = F.normalize(feats)
    return feats @ W.t() * head.s



    
# Objective function for Optuna to optimize hyperparameters
def objective(trial):
    accelerator = Accelerator()
    # Hyperparameters to optimize
    S_SCALE = trial.suggest_float('S_SCALE', 10.0, 50.0)  # Range of possible values
    M_MARGIN = trial.suggest_float('M_MARGIN', 0.2, 0.6)
    TAU = trial.suggest_float('TAU', 0.2, 0.9)

    df = pair_images_with_masks(IMAGE_DIR, MASK_DIR)
    if df.empty:
        raise RuntimeError("No image↔mask pairs found. Check names/rutas or adjust normalize_key().")

    # 2) Map labels from Excel
    label_map, label_col = read_labels_from_xls(XLS_PATH)
    df[label_col] = df["base"].map(label_map)
    miss = df[label_col].isna()
    if miss.any():
        df.loc[miss, label_col] = df.loc[miss, "base_norm"].map(label_map)
    df = df[~df[label_col].isna()].copy()
    #accelerator.print(f"[LABELS] after XLS join: {len(df)} rows")

    # 3) Filter classes
    inc = {p.strip().lower() for p in INCLUDE_PATTERNS.split(",") if p.strip()}
    df[label_col] = df[label_col].astype(str)
    df = df[df[label_col].str.lower().isin(inc)].copy()
    #accelerator.print(f"[FILTER] kept={len(df)} over classes={sorted(list(inc))}")
    if df.empty:
        raise RuntimeError("After filtering by classes, dataset is empty. Check INCLUDE_PATTERNS and Excel labels.")


    
    # 1. Build model with these hyperparameters
    classes = sorted(df[label_col].unique().tolist())
    label2id = {c:i for i,c in enumerate(classes)}
    id2label = {v:k for k,v in label2id.items()}
    model, _ = build_backbone(ARCH, pretrained=True, embed_dim=EMBED_DIM, in_channels=4)  # 4-channels if mask
    model = model.to(accelerator.device)
    loss_head = FuzzyArcLoss(EMBED_DIM, len(label2id), s=S_SCALE, m=M_MARGIN, tau=TAU)
    
    # 2. Optimizer
    optimizer = torch.optim.AdamW(list(model.parameters()) + list(loss_head.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)

    train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df[label_col])
    counts = Counter(train_df[label_col])
    # weights inversely proportional to class count
    ce_w = torch.tensor([1.0 / counts[id2label[i]] for i in range(len(id2label))], dtype=torch.float32, device=accelerator.device)


    
    # 3. Dataloader
    train_ds = HistMaskDataset(train_df, label_col, label2id, IMG_SIZE, True, USE_MASK_AS_CHANNEL, USE_ROI_CROP, ROI_PADDING, MASK_THRESH)
    val_ds = HistMaskDataset(val_df, label_col, label2id, IMG_SIZE, False, USE_MASK_AS_CHANNEL, USE_ROI_CROP, ROI_PADDING, MASK_THRESH)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    # 4. Train model
    model.train()
    accelerator = Accelerator()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for x, y, _ in train_loader:
        x, y = x.to(accelerator.device), y.to(accelerator.device)
        feats = model(x)
        loss, logits = loss_head(feats, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    
    # 5. Validation phase
    model.eval()
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for x, y, _ in val_loader:
            x, y = x.to(accelerator.device), y.to(accelerator.device)
            feats = model(x)
            logits = loss_head.head(feats, y)
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(y.cpu().numpy())

    # 6. Calculate metrics
    f1 = f1_score(val_labels, val_preds, average='macro')
    precision = precision_score(val_labels, val_preds, average='macro')
    recall = recall_score(val_labels, val_preds, average='macro')
    
    # Return the objective value (maximize F1 score)
    return f1  # or use a weighted average of precision and recall if desired
    # return (precision + recall) / 2  # you can adjust this to use precision and recall as well


In [17]:
# ---------------------
# Notebook launcher helper
# ---------------------
def launch_from_notebook(num_processes: int = NUM_PROCESSES):
    """Call this from Jupyter to spawn 6 processes."""
    return notebook_launcher(training_function, args=(), num_processes=num_processes, mixed_precision=MIXED_PRECISION)

def _in_notebook() -> bool:
    return 'ipykernel' in sys.modules


In [18]:

# Guard to avoid accidental single-GPU run when executed inside Jupyter
if __name__ == "__main__" and not _in_notebook():

    training_function()


In [19]:

import torch
print("CUDA devices visible:", torch.cuda.device_count())
# Optional: quick pairing check
# df = scan_and_sanity()
# df.head()


CUDA devices visible: 6


In [20]:

# Launch distributed training on 6 GPUs from this notebook
launch_from_notebook(num_processes=6)


Launching training on 6 CUDAs.
Running on 6 GPUs; mp=no
[SCAN] images=731 mats=731
[PAIRING] matched=731  unmatched=0
[LABELS] after XLS join: 731 rows
[FILTER] kept=637 over classes=['acinar', 'lepidic', 'micropapillary', 'mucinous', 'papillary', 'solid']
[CLASSES] {'acinar': 0, 'lepidic': 1, 'micropapillary': 2, 'mucinous': 3, 'papillary': 4, 'solid': 5}
[PAIRING] matched=731  unmatched=0
[PAIRING] matched=731  unmatched=0
[PAIRING] matched=731  unmatched=0
[PAIRING] matched=731  unmatched=0
[PAIRING] matched=731  unmatched=0
[001] tr_loss=0.1074 acc=0.0755 f1=0.0771 | val_loss=0.2293 acc=0.0521 f1=0.0483 | 10.6s
[OK] saved best -> /home/rapids/notebooks/slima/outputs/anorak_roi_acc6_v5/best_model_roi_acc6_v3.pth (f1=0.0483)
[002] tr_loss=0.0578 acc=0.3672 f1=0.3570 | val_loss=0.2168 acc=0.2500 f1=0.2155 | 8.7s
[OK] saved best -> /home/rapids/notebooks/slima/outputs/anorak_roi_acc6_v5/best_model_roi_acc6_v3.pth (f1=0.2155)
[003] tr_loss=0.0522 acc=0.4688 f1=0.4567 | val_loss=0.2200 a